# Kavach: Evon v3.3 server on Kaggle

Runs Gnani's **Evon v3.3** (4-bit GGUF, 23 GB) on Kaggle's free **2 × T4** GPUs with llama.cpp, and exposes it through a free Cloudflare tunnel so Kavach can call it.

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified Kaggle account). Then **Run All**. The first run takes about 20 to 25 minutes (llama.cpp build plus a 23 GB download).

At the end, the notebook prints three lines to paste into Kavach's `.env`.

In [ ]:
# 1. Settings. No edits needed: a new random API key is generated each run and printed at the end.
import secrets
API_KEY = "kavach-" + secrets.token_urlsafe(24)
MODEL_FILE = "gnani-evon-v3.3-30B-A3B-Q4_K_M.gguf"
MODEL_DIR = "/tmp/models"   # /kaggle/working is capped at about 20 GB
CONTEXT = 16384
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp | tail -1

In [ ]:
# 2. Build llama.cpp with CUDA for T4 (compute capability 7.5). About 10 to 15 minutes.
# Kaggle has no libcuda.so for linking, so CMake can't find CUDA::cuda_driver. llama.cpp only
# needs it for virtual memory management (VMM); GGML_CUDA_NO_VMM=ON skips that link.
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp 2>&1 | tail -1
!cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA_NO_VMM=ON -DCMAKE_LIBRARY_PATH=/usr/local/cuda/lib64/stubs > /tmp/cmake.log 2>&1 && cmake --build build -j4 --target llama-server > /tmp/build.log 2>&1; tail -3 /tmp/cmake.log; tail -3 /tmp/build.log
!ls -lh /tmp/llama.cpp/build/bin/llama-server

If the build or model load fails with an unknown architecture (`nemotron_h_moe`), the GGUF repo says its files were tested with llama.cpp commit `8b4b3558f1459c13e4aa38d5c94d306a00dc6acd`. Replace the clone in step 2 with:
`!git clone https://github.com/ggml-org/llama.cpp /tmp/llama.cpp && cd /tmp/llama.cpp && git checkout 8b4b3558f1459c13e4aa38d5c94d306a00dc6acd`

In [ ]:
# 3. Download Evon (about 23 GB, a few minutes). The repo is not gated, so no token is needed.
!pip -q install -U huggingface_hub
!hf download luminenio/gnani-evon-v3.3-30B-A3B-GGUF {MODEL_FILE} LICENSE LICENSE-NVIDIA NOTICE --local-dir {MODEL_DIR}
!ls -lh {MODEL_DIR}

In [ ]:
# 4. Start the OpenAI-compatible server on both GPUs.
import subprocess, time, urllib.request
server = subprocess.Popen(
    ["/tmp/llama.cpp/build/bin/llama-server", "-m", f"{MODEL_DIR}/{MODEL_FILE}",
     "-ngl", "99", "-c", str(CONTEXT), "--jinja", "--host", "0.0.0.0", "--port", "8080",
     "--api-key", API_KEY, "--parallel", "3"],
    stdout=open("/tmp/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://localhost:8080/health", timeout=2); print("server ready"); break
    except Exception:
        time.sleep(5)
else:
    print("server did not start; last log lines:")
!tail -5 /tmp/server.log

In [ ]:
# 5. Quick check: one Hindi scam snippet, thinking off, JSON out.
import json, time, urllib.request
body = {"model": "model", "temperature": 0.6, "top_p": 0.95, "max_tokens": 200,
        "chat_template_kwargs": {"enable_thinking": False},
        "messages": [{"role": "system", "content": "Reply only with JSON: {\"verdict\": \"SAFE\" or \"SCAM\", \"reason\": \"...\"}"},
                     {"role": "user", "content": "Caller: मैं सीबीआई से बोल रहा हूं, आप डिजिटल अरेस्ट में हैं, किसी को मत बताना, पैसे ट्रांसफर कीजिए"}]}
req = urllib.request.Request("http://localhost:8080/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
t0 = time.time(); reply = json.load(urllib.request.urlopen(req, timeout=120))
print(f"{time.time() - t0:.2f}s:", reply["choices"][0]["message"]["content"])

In [ ]:
# 6. Public HTTPS URL via a free Cloudflare quick tunnel (no account needed).
import re, subprocess, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8080"],
                          stdout=open("/tmp/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("Paste into Kavach's .env:")
print()
print(f"EVON_BASE_URL={url}/v1")
print("EVON_MODEL=model")
print(f"EVON_API_KEY={API_KEY}")


In [ ]:
# 7. Keep the session alive while you work (stop this cell to end). Prints health every minute.
import time, urllib.request
while True:
    try:
        urllib.request.urlopen("http://localhost:8080/health", timeout=5); status = "ok"
    except Exception as e:
        status = f"DOWN ({e})"
    print(time.strftime("%H:%M"), "server", status, flush=True)
    time.sleep(60)